In [ ]:
# 설치
!pip install anthropic

In [ ]:
# 예시 데이터
%%writefile notes.txt
12
7
30

In [ ]:
import anthropic
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

# 1) 툴 = 실제 함수 + 모델에게 보여줄 설명서
def calculator(a: float, op: str, b: float) -> str:
    OPS = {"+": lambda x, y: x + y, "-": lambda x, y: x - y,
           "*": lambda x, y: x * y, "/": lambda x, y: x / y}
    if op not in OPS:
        return "오류: 지원하지 않는 연산자"
    if op == "/" and b == 0:
        return "오류: 0으로 나눌 수 없음"
    return str(OPS[op](a, b))

def read_file(path: str) -> str:
    try:
        with open(path, encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"오류: {path} 파일이 없음"

tools = [
    {
        "name": "calculator",
        "description": "두 수를 연산(사칙연산)",
        "input_schema": {
            "type": "object",
            "properties": {
                            "a": {"type": "number"},
                            "op": {"type": "string", "enum":["+","-","*","/"]},
                            "b": {"type": "number"}
                        },
            "required": ["a", "op", "b"],
        }
    },
    {
        "name": "read_file",
        "description": "파일을 읽음",
        "input_schema": {
            "type": "object",
            "properties": {"path":{"type":"string"}},
            "required": ["path"],
        }
    }
]
TOOL_FUNCS = {"calculator": calculator, "read_file": read_file}

# 2) 대화 기록
messages = [{"role": "user", "content": "notes.txt 파일을 읽고 안쪽의 숫자들을 모두 더해줘."}]
last_call = None
# 3) 루프
for step in range(5):                                   # 최대 5바퀴
    response = client.messages.create(
        model="claude-sonnet-5-5", max_tokens=1024,
        tools=tools, messages=messages,
    )
    print(step, response.stop_reason, len(messages))
    messages.append({"role": "assistant", "content": response.content})  # (가) 모델 답 기록

    if response.stop_reason == "end_turn":       # 정상 종료만 최종 답으로
        print(next(b.text for b in response.content if b.type == "text"))
        break
    elif response.stop_reason != "tool_use":     # max_tokens 등은 따로 알림
        print(f"비정상 종료: {response.stop_reason}")
        break

    results = []
    stop = False
    for block in response.content:                      # (다) 툴 호출 처리
        if block.type == "tool_use":
            call = (block.name, block.input)
            if call == last_call:
                print(f"Error: {block.name} 연속 호출")
                stop = True; break

            last_call = call

            output = TOOL_FUNCS[block.name](**block.input)
            results.append({"type": "tool_result",
                            "tool_use_id": block.id,     # 어느 호출에 대한 답인지 짝 맞춤
                            "content": str(output)})
    if stop: break
    messages.append({"role": "user", "content": results})  # (라) 결과 기록 → 다음 바퀴
else: print("반복제한 초과로 종료")